# AeroExtract: Aligning Cohere Tiny Aya for Structured Aviation Maintenance Records
### Lexsi AI Safety & Alignment Hackathon | Track 1: Fine-tuning & Data Curation | Problem 5
**End-to-End Pipeline: CuratorKIT ➔ AlignTune ➔ AuditKIT**

This notebook trains a parameter-efficient LoRA adapter on `CohereLabs/tiny-aya-base-32K` to reliably extract strict, Pydantic-validated JSON from messy, handwritten aircraft maintenance logs.

## 1. Install Dependencies & Set Up Environment

In [ ]:
!pip install -q trl peft transformers datasets bitsandbytes accelerate pydantic

## 2. Load Model & Tokenizer (Tiny Aya 3.35B)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_ID = "CohereLabs/tiny-aya-base-32K"

# 4-bit quantization config for free Colab T4 / L4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully!")

## 3. Configure LoRA PEFT via AlignTune Specification

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Load CuratorKIT Dataset & Format Prompts

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "data/curated/train.jsonl",
    "validation": "data/curated/val.jsonl"
})

def format_prompt(example):
    text = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    return {"text": text}

formatted_dataset = dataset.map(format_prompt)
print(formatted_dataset["train"][0]["text"][:300])

## 5. SFT Post-Training with TRL

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./outputs/tiny_aya_aviation_extractor",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    dataset_text_field="text",
    max_seq_length=2048,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported()
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    args=training_args,
    peft_config=lora_config
)

trainer.train()
model.save_pretrained("./outputs/tiny_aya_aviation_adapter")
print("Training complete! Adapter saved to ./outputs/tiny_aya_aviation_adapter")